In [ ]:
!pip install anthropic
!pip install pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.5/239.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.8 MB/s eta 0:00:00


In [ ]:
import anthropic
import PyPDF2
import os
import re

from google.colab import userdata
API_KEY = userdata.get('claude-3.7-sonnet')

# Set up Claude API client
# client = anthropic.Anthropic()

client = anthropic.Anthropic(
    api_key=API_KEY,
)

# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    """Extracts text from a given PDF file."""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

# Function to analyze research papers with streaming
def analyze_papers_streaming(paper_texts):
    """Uses Claude 3.7 Sonnet with Thinking Mode in streaming mode to analyze papers, find drawbacks, and generate a research project."""
    with client.messages.stream(
        model="claude-3-7-sonnet-20250219",
        max_tokens=25000,
        thinking={
            "type": "enabled",
            "budget_tokens": 16000  # Large budget for deep reasoning
        },
        messages=[{
            "role": "user",
            "content": f"""
                You are an AI research assistant. Given the following research papers, perform the following tasks:

                1. **Summarize each paper** with its core contributions and findings.
                2. **Identify key drawbacks** of each paper, focusing on limitations, gaps, or areas needing improvement.
                3. **Find interconnections and citations** between the papers—what ideas, methods, or datasets do they share?
                4. **Propose a novel research idea** that addresses a major limitation across these papers.
                   - Suggest how techniques from different papers can be combined to solve a problem.
                   - Ensure the idea is practical and feasible for further research.
                   - Justify why this approach is promising.

                Research Papers:
                {paper_texts}
            """
        }]
    ) as stream:
        current_block_type = None
        current_content = ""

        for event in stream:
            if event.type == "content_block_start":
                current_block_type = event.content_block.type
                print(f"\n--- Starting {current_block_type} block ---")
                current_content = ""

            elif event.type == "content_block_delta":
                if event.delta.type == "thinking_delta":
                    print(event.delta.thinking, end="", flush=True)
                    current_content += event.delta.thinking
                elif event.delta.type == "text_delta":
                    print(event.delta.text, end="", flush=True)
                    current_content += event.delta.text

            elif event.type == "content_block_stop":
                if current_block_type == "thinking":
                    print(f"\n[Completed thinking block, {len(current_content)} characters]")
                elif current_block_type == "redacted_thinking":
                    print("\n[Redacted thinking block]")
                print(f"--- Finished {current_block_type} block ---\n")
                current_block_type = None

            elif event.type == "message_stop":
                print("\n--- Message complete ---")

# Example usage
def main():
    # Load research papers (example paths)
    pdf_paths = ["/content/paper1.pdf", "/content/paper2.pdf", "/content/paper3.pdf"]  # Add real paths

    # Extract text from all PDFs
    all_papers_text = "\n\n".join([extract_text_from_pdf(pdf) for pdf in pdf_paths])

    # Analyze papers using Claude 3.7 Sonnet with streaming
    analyze_papers_streaming(all_papers_text)

if __name__ == "__main__":
    main()



--- Starting thinking block ---
Let me analyze these three research papers on multimodal large language models (MLLMs). I'll summarize each one, identify their limitations, find interconnections, and propose a novel research idea.

# Paper 1: Florence-VL: Enhancing Vision-Language Models with Generative Vision Encoder and Depth-Breadth Fusion

## Summary:
Florence-VL is a new family of multimodal large language models (MLLMs) that uses Florence-2, a generative vision foundation model, as its visual encoder. Unlike CLIP-style encoders trained with contrastive learning, Florence-2 captures different levels and aspects of visual features. The authors propose:

1. A "depth-breath fusion" (DBFusion) architecture to fuse visual features extracted from different depths and under multiple prompts
2. A training recipe that integrates Florence-2's features into LLMs like Phi 3.5 and LLama 3
3. End-to-end pretraining followed by finetuning of projection layers and LLM on diverse datasets

The mo

# Gradio implementation

In [ ]:
!pip install gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 whic

In [ ]:
import anthropic
import PyPDF2
import os
import re
import gradio as gr

from google.colab import userdata
API_KEY = userdata.get('claude-3.7-sonnet')

# Set up Claude API client
# client = anthropic.Anthropic()

client = anthropic.Anthropic(
    api_key=API_KEY,
)
# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    """Extracts text from a given PDF file."""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text.strip()

# Function to analyze research papers with streaming
def analyze_papers_streaming(paper_texts, paper_count):
    """Uses Claude 3.7 Sonnet with Thinking Mode in streaming mode to analyze papers, find drawbacks, and generate a research project."""

    formatted_papers = "\n\n".join([f"### Paper {i+1}:\n{paper}" for i, paper in enumerate(paper_texts)])

    prompt = f"""
        You are an AI research assistant. You have been provided with {paper_count} research papers.
        Your task is to analyze these papers and perform the following:

        1. **Summarize each paper** with its core contributions and findings.
        2. **Identify key drawbacks** of each paper, focusing on limitations, gaps, or areas needing improvement.
        3. **Find interconnections and citations** between the papers—what ideas, methods, or datasets do they share?
        4. **Propose a novel research idea** that addresses a major limitation across these papers.
            - Suggest how techniques from different papers can be combined to solve a problem.
            - Ensure the idea is practical and feasible for further research.
            - Justify why this approach is promising.

        Below are the research papers:

        {formatted_papers}
    """

    results = {"thinking": "", "research_idea": ""}

    with client.messages.stream(
        model="claude-3-7-sonnet-20250219",
        max_tokens=25000,
        thinking={
            "type": "enabled",
            "budget_tokens": 16000  # Large budget for deep reasoning
        },
        messages=[{"role": "user", "content": prompt}]
    ) as stream:
        current_block_type = None

        for event in stream:
            if event.type == "content_block_start":
                current_block_type = event.content_block.type

            elif event.type == "content_block_delta":
                if event.delta.type == "thinking_delta":
                    results["thinking"] += event.delta.thinking
                elif event.delta.type == "text_delta":
                    results["research_idea"] += event.delta.text

            elif event.type == "message_stop":
                break

    return results["thinking"], results["research_idea"]

# Gradio UI function
def gradio_interface(pdfs):
    paper_texts = [extract_text_from_pdf(pdf.name) for pdf in pdfs]
    paper_count = len(paper_texts)  # Count number of provided papers
    thinking, research_idea = analyze_papers_streaming(paper_texts, paper_count)
    return thinking, research_idea

# Set up Gradio app
demo = gr.Interface(
    fn=gradio_interface,
    inputs=gr.File(file_types=[".pdf"], label="Upload Research Papers", file_count="multiple"),
    outputs=[gr.Textbox(label="Thinking Process"), gr.Textbox(label="Research Idea")],
    title="Claude 3.7 Sonnet - Powered Research Paper Analyzer",
    description="Upload multiple research papers to extract insights, identify key limitations, and generate a novel research idea."
)

if __name__ == "__main__":
    demo.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://934b00afd442fcc66c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://934b00afd442fcc66c.gradio.live
